# ESEF annual report anonymizer (pdf2htmlEX XHTML)

Workflow:
1. Install lxml once (first code cell).
2. In cell 0 set `INPUT_FILE` and `PAGES` (or leave `None` to be asked), then run cells 0-4 (setup and functions).
   The coefficient is asked at run time; never write it into the notebook.
3. Run cell 5: writes `candidates.csv` and `mapping_template.csv`.
4. Fill in `replacement` in `mapping_template.csv` (columns: original, replacement, type) and save it as `mapping.csv`.
   `mapping.csv`, `candidates.csv`, `output/` and the generated root `mapping_template.csv` are git-ignored.
5. Run cell 6: scales tagged amounts, removes signature blocks and repeated header/footer lines,
   replaces names, runs all checks and saves `output/anonymized.txt` + `output/key.json`.
6. Later (cell 7): `deanonymize()`, `unscale()`, `compare_numbers()`.

The original XHTML file is only read, never written. A filled-in mapping file is never overwritten.

In [ ]:
# Run once: installs lxml into this notebook's Python
%pip install lxml


In [ ]:
# 0. Imports and settings
import bisect
import copy
import csv
import html.entities
import io
import json
import re
import sys
import time
import unicodedata
from collections import Counter
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
from pathlib import Path

try:
    from lxml import etree
except ImportError as e:  # lxml handles big XHTML files and namespaces (ix:, xbrli:)
    raise ImportError("This script needs lxml:  pip install lxml") from e

# Leave as None to be asked interactively.
INPUT_FILE = None        # e.g. "data/FY2026_Kernel_Annual_Report.xhtml" (data/ is git-ignored)
PAGES = "114-118"        # decimal page numbers: "114-118", "135-136, 150" or [3, 114, 115]
COEFFICIENT = None       # private key: never write it here; asked at run time and saved to output/key.json

MAPPING_CSV = Path("mapping.csv")   # the filled-in copy of mapping_template.csv (git-ignored)
CANDIDATES_CSV = Path("candidates.csv")
MAPPING_TEMPLATE_CSV = Path("mapping_template.csv")
OUTPUT_DIR = Path("output")

MIN_WORD_COUNT = 3            # capitalized single words must occur at least this often
MIN_PHRASE_COUNT = 2          # capitalized multi-word phrases (e.g. "Kernel Holding S.A.")
PRINT_LIMIT = 40              # max rows printed per candidate category (CSV has everything)
SCALE_PER_SHARE_UNITS = True  # also scale USD/share (EPS, dividends per share); False = leave them
# Extra leak check after replacement: case-insensitive, no word boundaries. Surnames of `person` rows in
# mapping.csv are added automatically.
EXTRA_CHECK_TERMS = ["ker"]

# Page cleanup (done on the text, before names are replaced)
# Signature block: from a line starting "On behalf of the Board of Directors" (also "O" / "n behalf ..." split
# over two lines) up to and including the job-title line, searched within SIGNATURE_MAX_LINES lines.
SIGNATURE_START = r"(?im)^(?:O\s*)?n\s+behalf\s+of\s+the\s+Board"
SIGNATURE_END = r"(?i)Chief\s+Financial\s+Officer|\bCFO\b"            # preferred last line of the block
SIGNATURE_TITLES = r"(?i)Chairman|Chairwoman|Chief\s+\w+\s+Officer|\bCEO\b|\bCFO\b|\bDirector\b"
SIGNATURE_MAX_LINES = 15
# Header / footer / side menu: a line that occurs on every selected page (digits ignored, so
# "Annual Report 2026 | 114" and "... | 115" count as the same line). Needs 2+ pages.
SERVICE_LINE_KEEP = []        # lines that must never be removed, e.g. ["in thousands of US dollars"]
SERVICE_LINE_EXTRA = [r"^The accompanying notes are an integral part of these"]   # always removed (regex)

IN_NOTEBOOK = "ipykernel" in sys.modules
ISO4217_NS = "http://www.xbrl.org/2003/iso4217"
UP = "A-ZÀ-ÖØ-ÞА-ЯЁІЇЄҐ"      # upper-case letters (Latin + Cyrillic)


In [ ]:
# 1. Generic XML helpers
def localname(el):
    """Lower-case tag name without namespace or prefix ('ix:nonFraction' -> 'nonfraction')."""
    tag = el.tag
    if not isinstance(tag, str):          # comments / processing instructions
        return ""
    if "}" in tag:
        return tag.rsplit("}", 1)[1].lower()
    return tag.rsplit(":", 1)[-1].lower()


def namespace(el):
    """Namespace URI (or the prefix, if the file had to be parsed as plain HTML)."""
    tag = el.tag
    if isinstance(tag, str) and tag.startswith("{"):
        return tag[1:].split("}", 1)[0].lower()
    if isinstance(tag, str) and ":" in tag:
        return tag.split(":", 1)[0].lower()
    return ""


def is_ix(el, name):
    """True for an Inline XBRL element such as ix:header or ix:nonFraction."""
    ns = namespace(el)
    return localname(el) == name and ("inlinexbrl" in ns or ns == "ix")


def get_attr(el, name):
    """Attribute lookup that also works when the HTML parser lower-cased attribute names."""
    value = el.get(name)
    return el.get(name.lower()) if value is None else value


def drop(el):
    """Remove an element but keep the text that follows it (its tail)."""
    parent = el.getparent()
    if parent is None:
        return
    if el.tail:
        prev = el.getprevious()
        if prev is not None:
            prev.tail = (prev.tail or "") + el.tail
        else:
            parent.text = (parent.text or "") + el.tail
    parent.remove(el)


def load_tree(path):
    """Parse the XHTML read-only. Fallbacks: HTML named entities (&nbsp;) -> numeric, then recover mode, then HTML parser."""
    opts = dict(huge_tree=True, resolve_entities=False, no_network=True)
    try:
        return etree.parse(str(path), etree.XMLParser(**opts)), "xml"
    except etree.XMLSyntaxError as err:
        print(f"Strict XML parsing failed ({err}); trying fallbacks...")

    text = Path(path).read_bytes().decode("utf-8", errors="replace")
    keep = {"amp", "lt", "gt", "quot", "apos"}
    text = re.sub(
        r"&([A-Za-z][A-Za-z0-9]*);",
        lambda m: m.group(0) if m.group(1) in keep or m.group(1) not in html.entities.name2codepoint
        else f"&#{html.entities.name2codepoint[m.group(1)]};",
        text,
    )
    text = re.sub(r"^<\?xml[^>]*\?>", "", text.lstrip())   # we re-encode as UTF-8 ourselves
    data = text.encode("utf-8")
    for mode, parser in (("xml-entities-fixed", etree.XMLParser(**opts)),
                         ("xml-recover", etree.XMLParser(recover=True, **opts))):
        try:
            root = etree.fromstring(data, parser)
            if root is not None:
                return etree.ElementTree(root), mode
        except etree.XMLSyntaxError:
            pass
    root = etree.fromstring(data, etree.HTMLParser(huge_tree=True))
    return etree.ElementTree(root), "html"


In [ ]:
# 2. Load the report, read currency units, clean it, keep selected pages
DROP_TAGS = {"title", "meta", "script", "style", "img", "image", "link", "noscript"}


def parse_pages(raw):
    """'3, 114-116' -> [3, 114, 115, 116] (decimal page numbers, order kept, no duplicates)."""
    pages = []
    for part in re.split(r"[,;\s]+", str(raw).strip()):
        if not part:
            continue
        m = re.fullmatch(r"(\d+)\s*-\s*(\d+)", part)
        nums = range(int(m.group(1)), int(m.group(2)) + 1) if m else [int(part)]
        pages += [n for n in nums if n > 0 and n not in pages]
    if not pages:
        raise ValueError("No page numbers found")
    return pages


def find_currency_units(root):
    """Read xbrli:unit definitions (they live in ix:header, so do this BEFORE removing it)."""
    currency, per_share = set(), set()

    def is_currency_measure(m):
        prefix, _, _code = (m.text or "").strip().rpartition(":")
        uri = m.nsmap.get(prefix) if prefix and hasattr(m, "nsmap") else None
        return uri == ISO4217_NS or prefix.lower() == "iso4217"

    for unit in root.iter():
        if localname(unit) != "unit" or not unit.get("id"):
            continue
        measures = [m for m in unit.iter() if localname(m) == "measure"]
        has_divide = any(localname(d) == "divide" for d in unit.iter())
        if not has_divide and len(measures) == 1 and is_currency_measure(measures[0]):
            currency.add(unit.get("id"))            # e.g. USD
        elif has_divide and measures and is_currency_measure(measures[0]):
            per_share.add(unit.get("id"))           # e.g. USD / share (numerator is listed first)
    return currency, per_share


def guess_currency_unit(unit_id):
    """Fallback when no unit definitions were found: recognise ids like 'USD', 'U_USD', 'iso4217_EUR'."""
    if not unit_id or re.search(r"(?i)pure", unit_id):
        return False
    if re.search(r"(?i)share|per|/", unit_id) and not SCALE_PER_SHARE_UNITS:
        return False
    return bool(re.search(r"(?i)(?:^|[^a-z])(USD|EUR|UAH|GBP|PLN|CHF|JPY|CNY|SEK|NOK|DKK|CZK|HUF|RON|"
                          r"BGN|TRY|KZT|CAD|AUD|MDL|GEL)(?:$|[^a-z])", unit_id))


def prepare_document(input_file=None, pages=None):
    """Load file, remove header/meta/scripts/styles/images, return deep copies of the selected pages."""
    path = input_file or INPUT_FILE or input("Path to the XHTML file: ").strip().strip("'\"")
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)

    pages = pages or PAGES
    while not pages:
        try:
            pages = parse_pages(input("Page numbers in decimal (e.g. 3, 114, 115 or 114-116): "))
        except ValueError as err:
            print(err)
    pages = parse_pages(",".join(map(str, pages))) if not isinstance(pages, str) else parse_pages(pages)

    # Parse into memory. Nothing is ever written back to `path`.
    t0 = time.time()
    print(f"Reading {path.name} ({path.stat().st_size / 1e6:.0f} MB)...", flush=True)
    tree, mode = load_tree(path)
    print(f"  parsed in {time.time() - t0:.0f}s", flush=True)
    root = tree.getroot()

    # Units first: they are defined inside ix:header, which is removed next.
    currency_units, per_share_units = find_currency_units(root)

    # Remove ix:header, <title>, <meta>, <script>, <style> and images before any text is extracted.
    junk = [el for el in root.iter() if is_ix(el, "header") or localname(el) in DROP_TAGS]
    for el in junk:
        drop(el)
    print(f"  cleaned in {time.time() - t0:.0f}s", flush=True)

    # pdf2htmlEX pages are <div id="pfXX">, XX = page number in hexadecimal (114 -> pf72).
    page_by_id = {}
    for el in root.iter():
        pid = el.get("id") if isinstance(el.tag, str) else None
        if pid and re.fullmatch(r"pf[0-9a-fA-F]+", pid):
            page_by_id.setdefault(pid.lower(), el)
    selected, missing = {}, []
    for n in pages:
        el = page_by_id.get(f"pf{n:x}")
        if el is None:
            missing.append(n)
        else:
            selected[n] = copy.deepcopy(el)      # independent copy of just this page
    del tree, root, page_by_id                   # free memory of the full document

    if not selected:
        raise ValueError(f"None of the pages {pages} were found (looked for ids like pf{pages[0]:x}).")

    n_facts = sum(1 for el in selected.values() for d in el.iter() if is_ix(d, "nonfraction"))
    print(f"Parsed {path.name} ({mode}). Pages kept: {list(selected)}"
          + (f" | NOT FOUND: {missing}" if missing else ""))
    print(f"ix:nonFraction facts on these pages: {n_facts}")
    print(f"Currency units: {sorted(currency_units) or 'none found (fallback: guess from unitRef)'}"
          f" | per-share units: {sorted(per_share_units) or '-'}")
    return {"source": str(path.resolve()), "pages": selected,
            "currency_units": currency_units, "per_share_units": per_share_units}


In [ ]:
# 3. Text extraction (pdf2htmlEX layout -> plain text lines)
# pdf2htmlEX splits words and numbers into many <span>s ("Ker<span class="_ _3"></span>nel"), so spans add
# nothing; only <div> (one text line each) starts a new line.
BLOCK_TAGS = {"div"}
INVISIBLE = dict.fromkeys(map(ord, "­​‌‍⁠﻿"), None)


def normalize_inline(s):
    """NFKC (ligatures 'ﬁ'->'fi', NBSP->space), drop zero-width chars, collapse spaces."""
    s = unicodedata.normalize("NFKC", s).translate(INVISIBLE)
    return re.sub(r"[^\S\n]+", " ", s)


def normalize_text(s):
    lines = (ln.strip() for ln in normalize_inline(s).split("\n"))
    return "\n".join(ln for ln in lines if ln)


def _emit(el, out):
    name = localname(el)
    if name in BLOCK_TAGS:
        out.append("\n")                      # each pdf2htmlEX text line is a <div class="t ...">
    if el.text:
        out.append(el.text)                   # spans contribute only their own text, no separator
    for child in el:
        if isinstance(child.tag, str):
            _emit(child, out)
        if child.tail:
            out.append(child.tail)
    if name in BLOCK_TAGS:
        out.append("\n")


def page_texts(pages):
    texts = {}
    for n, el in pages.items():
        out = []
        _emit(el, out)
        texts[n] = normalize_text("".join(out))
    return texts


def assemble_text(texts):
    return "\n\n".join(f"=== Page {n} ===\n{t}" for n, t in texts.items())


In [ ]:
# 3b. Line-break hyphenation
HYPHEN_BREAK_RE = re.compile(r"(?<=[^\W\d_])[-‐‑]\n(?=[^\W\d_])")


def join_hyphenation(texts):
    """'Ker-\nnel' -> 'Kernel': a line ending in a hyphen after a letter, next line starting with a letter."""
    out, total = {}, 0
    for n, t in texts.items():
        out[n], k = HYPHEN_BREAK_RE.subn("", t)
        total += k
    return out, total


In [ ]:
# 3c. Page cleanup - signature blocks and repeated header / footer / menu lines
def remove_signature_blocks(texts, verbose=True):
    """Delete 'On behalf of the Board of Directors' ... job-title line on every page."""
    start_rx, end_rx, title_rx = re.compile(SIGNATURE_START), re.compile(SIGNATURE_END), re.compile(SIGNATURE_TITLES)
    out = {}
    for n, t in texts.items():
        lines = t.split("\n")
        while True:
            m = start_rx.search("\n".join(lines))
            if not m:
                break
            start = "\n".join(lines)[:m.start()].count("\n")           # line where the block starts
            window = range(start + 1, min(len(lines), start + 1 + SIGNATURE_MAX_LINES))
            ends = [i for i in window if end_rx.search(lines[i])] or [i for i in window if title_rx.search(lines[i])]
            if not ends:
                print(f"  ! p.{n}: signature start found but no job-title line within "
                      f"{SIGNATURE_MAX_LINES} lines - left as is: {lines[start][:60]!r}")
                break
            end = ends[-1]
            removed = lines[start:end + 1]
            if verbose:
                print(f"  p.{n}: removed signature block ({len(removed)} lines): "
                      f"{removed[0][:40]!r} ... {removed[-1][:60]!r}")
            del lines[start:end + 1]
        out[n] = "\n".join(lines)
    return out


def _service_key(line):
    """Comparison key: case-insensitive, digits ignored. Lines with < 2 letters (numbers) never qualify."""
    if len(re.findall(r"[^\W\d_]", line)) < 2:
        return None
    return re.sub(r"\d+", "#", " ".join(line.split())).casefold()


def remove_service_lines(texts, verbose=True):
    """Delete lines that occur on every selected page, plus SERVICE_LINE_EXTRA. Returns (texts, removed)."""
    keep = {_service_key(k) for k in SERVICE_LINE_KEEP}
    extra = [re.compile(p, re.I) for p in SERVICE_LINE_EXTRA]
    if len(texts) < 2:
        if verbose:
            print("  ! only one page selected - automatic header/footer detection skipped")
        service = set()
    else:
        on_pages = Counter()
        for t in texts.values():
            on_pages.update({k for k in map(_service_key, t.split("\n")) if k})
        service = {k for k, c in on_pages.items() if c == len(texts) and k not in keep}
    out, removed = {}, Counter()
    for n, t in texts.items():
        kept = []
        for line in t.split("\n"):
            k = _service_key(line)
            if k in service or (k and k not in keep and any(rx.search(line) for rx in extra)):
                removed[line] += 1
            else:
                kept.append(line)
        out[n] = "\n".join(kept)
    return out, removed


def clean_pages(texts, verbose=True):
    """Signature blocks first (their lines can repeat on every page too), then header/footer/menu lines."""
    if verbose:
        print("\nPage cleanup - signature blocks")
    texts = remove_signature_blocks(texts, verbose)
    texts, removed = remove_service_lines(texts, verbose)
    if verbose:
        print(f"Page cleanup - {len(removed)} distinct header/footer/menu lines removed (× = times removed):")
        for line, c in sorted(removed.items(), key=lambda x: -x[1]):
            print(f"  {c:>3} ×  {line[:100]}")
    return texts


def prepared_texts(pages, verbose=True):
    """Text used for candidates, replacement and checks: extraction -> hyphenation joined -> cleanup."""
    texts, joined = join_hyphenation(page_texts(pages))
    if verbose:
        print(f"\nJoined {joined} words hyphenated across line breaks (e.g. 'Ker-' + 'nel' -> 'Kernel')")
    return clean_pages(texts, verbose)


In [ ]:
# 4a. Step 1 - candidate detection (nothing is replaced here)
STOPWORDS = set("""a an the this that these those there their they them it its in on at for of and or nor but
to from with without by as is are was were be been being if when where while which who whom whose what why how
all any each every some such no not our ours we us you your he she his her also other others than then thus
therefore however further furthermore moreover during before after under over between into through per within
against including based note notes total totals table page year years ended ending period annual report
financial statement statements consolidated separate interim group company companies management section part
january february march april may june july august september october november december monday tuesday wednesday
thursday friday saturday sunday ifrs ias eu us usd eur uah mr mrs ms dr sir board director directors chairman
ceo cfo coo chief executive officer officers independent non-executive""".split())

CAPTOK = rf"[{UP}][\w\-&'’.]*"
NAMETOK = rf"(?:[{UP}][a-zà-ÿа-яёіїєґ'’\-]+|[{UP}]{{2,}}|[{UP}]\.)"
NAME2 = rf"{NAMETOK}(?: {NAMETOK}){{1,3}}"          # 2-4 name-like words on one line
HONORIFIC = r"(?:Mr|Mrs|Ms|Miss|Mx|Sir|Dame|Dr|Prof|Lord|Lady)\.?"
ROLE = (r"(?:CEO|CFO|COO|CTO|CIO|CRO|Chief [A-Z][a-z]+(?: [A-Z][a-z]+)? Officer|Chairman|Chairwoman|"
        r"Chairperson|Chair|(?:Non-[Ee]xecutive |Independent |Executive |Managing )?Director|Board|"
        r"President|Company Secretary)")
LEGAL = (r"(?:S\.A\.|S\.à r\.l\.|SARL|LLC|L\.L\.C\.|Ltd\.?|Limited|PLC|plc|PJSC|PrJSC|JSC|OJSC|CJSC|LLP|"
         r"B\.V\.|N\.V\.|GmbH|AG|SE|Inc\.?|Corp\.?|Corporation|S\.p\.A\.|Sp\. z o\.o\.|TOV|ТОВ|ПрАТ|ПАТ|АТ|ТзОВ)")

ASSET = (r"(?:Terminals?|Plants?|Elevators?|Ports?|Farms?|Silos?|Hubs?|Brands?|Fleet|Vessels?|Railcars|"
         r"Facilit(?:y|ies)|Stations?|Complex|Mills?|Warehouses?)")
REGEX_DETECTORS = [
    # (category, compiled regex, group to take)
    ("name next to title", re.compile(rf"\b{HONORIFIC} ({NAMETOK}(?: {NAMETOK}){{0,3}})"), 1),
    ("name next to title", re.compile(rf"\b{ROLE}\b(?:,|:| -| –| —)?\s+(?:of the Board\s+)?({NAME2})"), 1),
    ("name next to title", re.compile(rf"({NAME2})(?:,| -| –| —| \(|:)?\s+(?:the )?{ROLE}\b"), 1),
    ("company (legal form)", re.compile(rf"((?:{CAPTOK} ){{1,5}}{LEGAL})(?!\w)"), 1),
    ("company (legal form)", re.compile(rf"((?:LLC|PJSC|PrJSC|JSC|OJSC|CJSC|TOV|ТОВ|ПрАТ|ПАТ|АТ|ТзОВ) "
                                        rf"[«\"“]?{CAPTOK}(?: {CAPTOK}){{0,3}}[»\"”]?)"), 1),
    ("product / asset name", re.compile(rf"((?:{CAPTOK} ){{1,3}}(?:{ASSET}|{ASSET.lower()}))(?!\w)"), 1),
    ("product / asset name", re.compile(rf"[«“\"]([{UP}][^»”\"\n]{{1,40}}?)[»”\"]"), 1),
    ("product / asset name", re.compile(rf"({CAPTOK}(?: {CAPTOK}){{0,3}})\s?[®™]"), 1),
    ("product / asset name", re.compile(r"\b([A-Z][a-z]+[A-Z][A-Za-z]+)\b"), 1),   # CamelCase / glued words
    ("email", re.compile(r"[\w.+\-]+@[\w\-]+(?:\.[\w\-]+)+"), 0),
    ("url", re.compile(r"(?i)\b(?:https?://|www\.)[^\s<>\"'()]+"), 0),
    ("url", re.compile(r"(?i)\b[\w\-]+(?:\.[\w\-]+)*\.(?:com|net|org|ua|lu|eu|info|biz|io|co|uk|de|pl|cy|nl|ch)\b"
                       r"(?:/[^\s<>\"'()]*)?"), 0),
    ("code (LEI-like, 20 chars)", re.compile(r"\b[A-Z0-9]{18}[0-9]{2}\b"), 0),
    ("code (ISIN-like)", re.compile(r"\b[A-Z]{2}[A-Z0-9]{9}[0-9]\b"), 0),
    ("code (letters+digits, 8+)", re.compile(r"\b(?=[A-Z0-9]*\d)(?=[A-Z0-9]*[A-Z])[A-Z0-9]{8,}\b"), 0),
    ("code (registration-like)", re.compile(r"\b[A-Z]{1,3}[ \-]?\d{5,}\b"), 0),
    ("long number (8+ digits)", re.compile(r"(?<![\d,.])\d{8,}(?![\d,.])"), 0),
    ("phone", re.compile(r"\+\d{1,3}[\s\-()]*\d(?:[\s\-()]*\d){6,12}"), 0),
]
CATEGORY_TYPE = {"name next to title": "person", "product / asset name": "product", "company (legal form)": "company", "email": "email",
                 "url": "url", "capitalized phrase": "", "capitalized word": ""}


def _clean_candidate(s):
    s = re.sub(r"['’]s$", "", s.strip().strip(",;:()[]«»\"“”"))
    if s.endswith(".") and s.count(".") == 1:     # sentence full stop, not an abbreviation like S.A.
        s = s[:-1]
    return s.rstrip(".,;:") if s.lower().startswith(("http", "www")) else s


def _context(text, start, end, width=40):
    return text[max(0, start - width):end + width].replace("\n", " / ")


def _word_rx(s):
    return re.compile(rf"(?<!\w){re.escape(s)}(?!\w)")


def find_candidates(texts):
    text = assemble_text(texts)
    found = {}   # (category, candidate) -> [count, first_position]

    def add(cat, cand, pos, inc=1):
        if not cand or (cat.startswith(("name", "product")) and all(t.lower().strip(".") in STOPWORDS for t in cand.split())):
            return
        rec = found.setdefault((cat, cand), [0, pos])
        rec[0] += inc

    # Regex-based detectors: titles, legal forms, URLs, emails, codes, phones.
    seen_codes, seen_pos = set(), set()
    for cat, rx, grp in REGEX_DETECTORS:
        for m in rx.finditer(text):
            if (cat, m.start(grp)) in seen_pos:
                continue                          # same hit already found by another regex of this category
            seen_pos.add((cat, m.start(grp)))
            cand = _clean_candidate(m.group(grp))
            if cat.startswith(("code", "long number")):
                if cand in seen_codes and (cat, cand) not in found:
                    continue                      # already listed under a more specific code category
                seen_codes.add(cand)
            add(cat, cand, m.start(grp))

    # Capitalized single words occurring MIN_WORD_COUNT+ times ("Kernel's" counts as "Kernel").
    words = Counter(w.strip("-") for w in re.findall(r"(?<![\w\-])[^\W\d_][\w\-]*", text))
    for w, c in words.items():
        if c >= MIN_WORD_COUNT and len(w) > 1 and w[0].isupper() and w.lower() not in STOPWORDS:
            add("capitalized word", w, text.find(w), c)

    # Capitalized phrases (2-5 words) - every n-gram inside runs of capitalized words, then prune
    # sub-phrases that never occur outside a longer phrase.
    grams = Counter()
    for line in text.split("\n"):
        for run in re.finditer(rf"(?<![\w\-]){CAPTOK}(?: {CAPTOK})+", line):
            toks = [_clean_candidate(t) for t in run.group(0).split(" ")]
            for n in range(2, min(5, len(toks)) + 1):
                for i in range(len(toks) - n + 1):
                    g = toks[i:i + n]
                    if g[0].lower() not in STOPWORDS and g[-1].lower() not in STOPWORDS:
                        grams[" ".join(g)] += 1
    kept = {g: c for g, c in grams.items() if c >= MIN_PHRASE_COUNT}
    covered = set()                                   # sub-phrases with the same count as a longer phrase
    for o, c in kept.items():
        toks = o.split(" ")
        for n in range(2, len(toks)):
            for i in range(len(toks) - n + 1):
                sub = " ".join(toks[i:i + n])
                if kept.get(sub) == c:
                    covered.add(sub)
    for g, c in kept.items():
        if g not in covered:
            add("capitalized phrase", g, text.find(g), c)

    # Build rows: count, pages where it occurs, short context of the first occurrence.
    rows = []
    for (cat, cand), (count, pos) in found.items():
        pages = [str(n) for n, t in texts.items() if cand in t]    # plain substring: fast
        rows.append({"category": cat, "candidate": cand, "count": count, "pages": " ".join(pages),
                     "context": _context(text, pos, pos + len(cand))})
    order = [c for c, _, _ in REGEX_DETECTORS] + ["capitalized phrase", "capitalized word"]
    order = list(dict.fromkeys(order))
    rows.sort(key=lambda r: (order.index(r["category"]), -r["count"], r["candidate"]))
    return rows


def _has_replacements(path):
    """True if a mapping/template CSV exists and at least one row has a replacement filled in."""
    try:
        raw = Path(path).read_text(encoding="utf-8-sig")
        dialect = csv.Sniffer().sniff(raw.splitlines()[0], delimiters=",;\t")
        rows = csv.DictReader(io.StringIO(raw), dialect=dialect)
        return any((r.get("replacement") or "").strip() for r in rows)
    except (OSError, IndexError, csv.Error):
        return False


def run_step1(doc):
    """Step 1: print candidates and save candidates.csv + mapping_template.csv. Nothing is replaced."""
    t0 = time.time()
    print("Step 1 - searching for candidates...", flush=True)
    rows = find_candidates(prepared_texts(doc["pages"], verbose=False))
    print(f"  done in {time.time() - t0:.0f}s", flush=True)
    current = None
    for r in rows:
        if r["category"] != current:
            current = r["category"]
            group = [x for x in rows if x["category"] == current]
            print(f"\n## {current} ({len(group)})")
            shown = 0
        if shown < PRINT_LIMIT:
            print(f"{r['count']:>4}  {r['candidate'][:45]:<45}  …{r['context']}…")
        elif shown == PRINT_LIMIT:
            print("      ... more in candidates.csv")
        shown += 1

    # utf-8-sig so Excel opens Cyrillic/diacritics correctly
    with open(CANDIDATES_CSV, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=["category", "candidate", "count", "pages", "context"])
        w.writeheader()
        w.writerows(rows)
    # A starting point for mapping.csv: copy the rows you need and fill in `replacement`.
    template = MAPPING_TEMPLATE_CSV
    if _has_replacements(template):                 # never overwrite a template you already filled in
        template = template.with_name(template.stem + "_new.csv")
        print(f"\n{MAPPING_TEMPLATE_CSV} already has replacements filled in - not overwritten; "
              f"new candidates go to {template}")
    with open(template, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.writer(f)
        w.writerow(["original", "replacement", "type"])
        for cand in dict.fromkeys(r["candidate"] for r in rows):
            cat = next(r["category"] for r in rows if r["candidate"] == cand)
            w.writerow([cand, "", CATEGORY_TYPE.get(cat, "code")])
    print(f"\nSaved {len(rows)} candidates to {CANDIDATES_CSV} and a template to {template}.")
    print(f"Now fill in {MAPPING_CSV} (original, replacement, type) and run steps 2-4.")
    return rows


In [ ]:
# 4b. Step 2 - name replacement
def read_mapping(path=MAPPING_CSV):
    """Read mapping.csv (comma, semicolon or tab separated; Excel BOM ok)."""
    raw = Path(path).read_text(encoding="utf-8-sig")
    try:
        dialect = csv.Sniffer().sniff(raw.splitlines()[0], delimiters=",;\t")
    except (csv.Error, IndexError):
        dialect = csv.excel
    reader = csv.DictReader(io.StringIO(raw), dialect=dialect)
    reader.fieldnames = [h.strip().lower() for h in reader.fieldnames or []]
    if not {"original", "replacement"} <= set(reader.fieldnames):
        raise ValueError(f"{path} needs columns original, replacement, type - got {reader.fieldnames}")

    mapping, seen = [], set()
    for i, row in enumerate(reader, start=2):
        orig = normalize_inline(row.get("original") or "").strip()
        repl = normalize_inline(row.get("replacement") or "").strip()
        if not orig:
            continue
        if not repl:
            print(f"  ! line {i}: '{orig}' has no replacement - skipped")
            continue
        if orig in seen:
            print(f"  ! line {i}: duplicate original '{orig}' - first one kept")
            continue
        seen.add(orig)
        mapping.append({"original": orig, "replacement": repl, "type": (row.get("type") or "").strip()})

    # Warnings that would break the leak check or reversibility.
    for m in mapping:
        for o in mapping:
            if o["original"].lower() in m["replacement"].lower():
                print(f"  ! replacement '{m['replacement']}' contains original '{o['original']}' "
                      f"- the leak check will flag it")
    for m in mapping:
        if len(m["replacement"]) < 3:
            print(f"  ! '{m['original']}' → '{m['replacement']}': replacement shorter than 3 characters - "
                  f"too close to the original name, use something like 'Company A'")
        elif m["replacement"].lower() in m["original"].lower():
            print(f"  ! '{m['original']}' → '{m['replacement']}': replacement is part of the original")
        for problem in pseudonym_problems(m["original"], m["replacement"]):
            print(f"  ! '{m['original']}' → '{m['replacement']}': {problem} - use a neutral ID "
                  f"(Company A, Person 1)")
    dup = [r for r, c in Counter(m["replacement"] for m in mapping).items() if c > 1]
    if dup:
        print(f"  ! same replacement used for several originals: {dup} - deanonymize() restores the longest original")
    return mapping


def pseudonym_problems(original, replacement):
    """Replacement tokens that hint at the original: its first letter alone, its initials, or its beginning."""
    words = re.findall(r"[^\W\d_]+", original)
    if not words:
        return []
    first, initials = words[0][0].lower(), "".join(w[0] for w in words).lower()
    problems = []
    for tok in re.findall(r"[^\W\d_]+", replacement):
        t = tok.lower()
        if len(t) == 1 and t == first:
            problems.append(f"'{tok}' is the first letter of the original")
        elif len(t) >= 2 and len(initials) >= 2 and (t.startswith(initials) or initials.startswith(t)):
            problems.append(f"'{tok}' looks like the initials ({initials.upper()})")
        elif len(t) >= 2 and t not in GENERIC_WORDS | {"person", "people"} and any(w.lower().startswith(t) for w in words):
            problems.append(f"'{tok}' is the beginning of the original")
    return problems


def build_name_regex(strings):
    """One regex for all strings: longest first, case-sensitive, whole words, any whitespace between words,
    optional possessive 's / ’s kept as is."""
    alts = [r"\s*".join(map(re.escape, s.split())) for s in sorted(strings, key=len, reverse=True)]
    return re.compile(r"(?<!\w)(" + "|".join(alts) + r")(['’]s)?(?!\w)")


def _nows(s):
    """Key without whitespace, so 'Kernel Holding', 'Kernel\nHolding' and 'KernelHolding' are the same."""
    return re.sub(r"\s+", "", s)


def replace_names(text, mapping):
    """Single pass, so text that was already replaced is never replaced again."""
    lookup = {}
    for m in mapping:
        lookup.setdefault(_nows(m["original"]), m)
    counts = Counter()

    def _sub(mo):
        m = lookup[_nows(mo.group(1))]
        counts[m["original"]] += 1
        return m["replacement"] + (mo.group(2) or "")

    return build_name_regex(m["original"] for m in mapping).sub(_sub, text), counts


In [ ]:
# 4c. Step 3 - scale currency amounts inside ix:nonFraction
NUM_CORE_RE = re.compile(r"\d(?:[\d,.'\s   ]*\d)?")


def _group3(digits, sep):
    parts = []
    while len(digits) > 3:
        parts.insert(0, digits[-3:])
        digits = digits[:-3]
    return sep.join([digits] + parts)


def scale_display(text, fmt, scale, sign, coef):
    """Return (new_text, info) or (None, reason). Keeps separators, decimals, parentheses, dashes."""
    fmt = (fmt or "").rsplit(":", 1)[-1].lower()
    if re.search(r"zero|dash|nocontent|empty", fmt):
        return None, "dash / fixed-zero"
    m = NUM_CORE_RE.search(text)
    if not m:
        return None, "no digits (e.g. '—')"
    if re.search(r"\d", text[:m.start()] + text[m.end():]):
        return None, "more than one number in tag"

    # Which character is the decimal separator?
    if not fmt:
        kind = "plain"
    elif re.search(r"comma-?decimal", fmt) or fmt in ("numdotcomma", "numspacecomma"):
        kind = "comma"
    elif re.search(r"dot-?decimal", fmt) or fmt in ("numcommadot", "numspacedot"):
        kind = "dot"
    else:
        return None, f"unsupported format '{fmt}'"
    dec_char = "," if kind == "comma" else "."

    # `text` is the whole tag content (all nested spans joined). pdf2htmlEX can leave a stray space inside
    # ("369,47 0"): remove whitespace, unless it is the real thousand separator ("1 194 387").
    core = m.group(0)
    ws = "[\\s\u00a0\u202f\u2009]"
    if not re.fullmatch(rf"\d{{1,3}}(?:{ws}\d{{3}})+(?:[.,]\d+)?", core):
        core = re.sub(ws, "", core)
    int_part, _, frac = core.rpartition(dec_char) if dec_char in core else (core, "", "")
    seps = set(re.sub(r"\d", "", int_part))
    if (kind == "plain" and seps) or len(seps) > 1 or not frac.isdigit() and frac:
        return None, f"unparseable '{core}'"
    sep = seps.pop() if seps else ""
    places = len(frac)

    # True value = sign × displayed × 10^scale; scale the true value, then convert back to display units.
    displayed = Decimal(re.sub(r"\D", "", int_part) + ("." + frac if frac else ""))
    factor = Decimal(10) ** int(scale or 0)
    signum = -1 if sign == "-" else 1
    true_value = signum * displayed * factor
    new_true = true_value * coef
    new_displayed = (abs(new_true) / factor).quantize(Decimal(1).scaleb(-places), rounding=ROUND_HALF_UP)

    # Rebuild with the same look (thousand separator, decimal places).
    int_str, _, frac_str = f"{new_displayed:f}".partition(".")
    if not sep and kind != "plain" and len(int_str) > 3:
        sep = "," if kind == "dot" else "."
    new_core = (_group3(int_str, sep) if sep else int_str) + (dec_char + frac_str if places else "")
    info = {"original_text": normalize_inline(core), "scaled_text": normalize_inline(new_core),
            "original_value": str(true_value), "scaled_value": str(new_true)}
    return (text[:m.start()] + new_core + text[m.end():]).strip(), info


def _text_slots(el):
    yield el, "text"
    for child in el:
        yield from _text_slots(child)
        yield child, "tail"


def set_element_text(el, new_text):
    """Put the new text in the first non-empty text slot inside the tag and empty the others."""
    slots = [(e, a) for e, a in _text_slots(el) if (e is not el or a == "text") and getattr(e, a)]
    if not slots:
        el.text = new_text
        return
    for i, (e, attr) in enumerate(slots):
        setattr(e, attr, new_text if i == 0 else "")


def scale_numbers(pages, coef, currency_units, per_share_units):
    """Multiply only ix:nonFraction amounts whose unitRef is a currency. Works on the given (copied) pages."""
    records, skipped = [], Counter()
    for n, page in pages.items():
        for el in page.iter():
            if not is_ix(el, "nonfraction"):
                continue
            if any(is_ix(d, "nonfraction") for d in el.iterdescendants()):
                continue                                   # nested wrapper: the inner tag is handled
            unit = get_attr(el, "unitRef") or ""
            is_cur = unit in currency_units or (not currency_units and guess_currency_unit(unit))
            if not (is_cur or (SCALE_PER_SHARE_UNITS and unit in per_share_units)):
                skipped[f"unit '{unit}' is not a currency"] += 1    # shares, %, ratios, per-share ...
                continue
            text = "".join(el.itertext())
            new_text, info = scale_display(text, get_attr(el, "format"), get_attr(el, "scale"),
                                           get_attr(el, "sign"), coef)
            if new_text is None:
                skipped[info] += 1
                continue
            set_element_text(el, new_text)
            info.update(page=n, concept=get_attr(el, "name"), unit=unit, scale=get_attr(el, "scale"),
                        sign=get_attr(el, "sign"), format=get_attr(el, "format"))
            records.append(info)
    return records, skipped


def ask_coefficient(value=None):
    raw = value if value is not None else COEFFICIENT
    while True:
        if raw is None:
            raw = input("Coefficient to multiply currency amounts by (e.g. 1.37): ")
        try:
            c = Decimal(str(raw).strip().replace(",", "."))
        except InvalidOperation:
            c = None
        if c is not None and c.is_finite() and c > 0:
            break
        print("Please enter a positive number, e.g. 1.37")
        raw = None
    if c == 1:
        print("  ! coefficient is 1 - amounts will not change")
    return c


In [ ]:
# 4d. Step 4 - leak check, save, deanonymize
def report_matches(text, terms, width=60):
    """Case-insensitive, no word boundaries, any whitespace between words. Prints every hit with context."""
    marks = [(m.start(), m.group(1)) for m in re.finditer(r"^=== Page (\d+) ===$", text, re.M)]
    starts = [p for p, _ in marks]
    total = 0
    for term in terms:
        rx = re.compile(r"\s*".join(map(re.escape, term.split())), re.I)
        for hit in rx.finditer(text):
            total += 1
            page = marks[bisect.bisect_right(starts, hit.start()) - 1][1] if marks else "?"
            ctx = text[max(0, hit.start() - width):hit.end() + width].replace("\n", " / ")
            print(f"  LEFT  '{term}'  [p.{page}]  …{ctx}…")
    return total


def leak_check(text, mapping):
    """Every original string from mapping.csv must be gone."""
    total = report_matches(text, [m["original"] for m in mapping])
    print(f"Leak check: {total} remaining match(es) (expected 0).")
    return total


def extra_term_check(text, mapping):
    """Surnames alone ('Verevskyi') and EXTRA_CHECK_TERMS ('kernel'), also inside other words."""
    surnames = [m["original"].split()[-1] for m in mapping
                if m.get("type", "").lower() in ("person", "people", "name") and len(m["original"].split()) > 1]
    if not surnames:
        print("  ! no multi-word rows with type=person in mapping.csv - surnames cannot be checked alone")
    surnames = list(dict.fromkeys(surnames))
    total = report_matches(text, surnames)
    print(f"Surnames alone ({', '.join(surnames) or '-'}): {total} match(es) (expected 0).")
    terms = list(dict.fromkeys(EXTRA_CHECK_TERMS))
    n = report_matches(text, terms)
    print(f"Extra terms ({', '.join(terms)}, case-insensitive, inside words too): {n} match(es).")
    return total + n


GENERIC_WORDS = STOPWORDS | set("""holding holdings group limited company companies corporation international
trading trade investments capital services management agro""".split())


def squashed_leak_check(text, mapping, width=60):
    """Copy of each page with all hyphens and line breaks removed; search every original and the first
    5 letters of each of its words (e.g. 'Kerne', 'Verev'), case-insensitive."""
    terms = []
    for m in mapping:
        terms.append(m["original"])
        terms += [w[:5] for w in re.findall(r"[^\W\d_]+", m["original"])
                  if len(w) >= 5 and w.lower() not in GENERIC_WORDS]
    terms = list({t.lower(): t for t in terms}.values())
    parts = re.split(r"(?m)^=== Page (\d+) ===$", text)
    total = 0
    for n, body in zip(parts[1::2], parts[2::2]):
        sq = re.sub(r"[-‐‑\n]", "", body)
        for term in terms:
            rx = re.compile(r"\s*".join(map(re.escape, term.split())), re.I)
            for hit in rx.finditer(sq):
                total += 1
                print(f"  LEFT  '{term}'  [p.{n}]  …{sq[max(0, hit.start() - width):hit.end() + width]}…")
    print(f"Squashed-text check (no hyphens / line breaks; originals + 5-letter prefixes "
          f"{', '.join(t for t in terms if len(t) == 5)}): {total} match(es) (expected 0).")
    return total


def unmapped_candidates(texts, mapping, save_to=Path("candidates_unmapped.csv")):
    """Step 1 again on the selected pages: capitalized words/phrases, names, product/asset names whose text
    contains no mapping.csv original (so they will NOT be replaced)."""
    origs = [m["original"].lower() for m in mapping]
    cats = ("name next to title", "company (legal form)", "product / asset name",
            "capitalized phrase", "capitalized word")

    def covered(cand):
        c = cand.lower()
        return any(o == c or re.search(rf"(?<!\w){re.escape(o)}(?!\w)", c) for o in origs)

    rows = [r for r in find_candidates(texts) if r["category"] in cats and not covered(r["candidate"])]
    print(f"\nCandidates on the selected pages that mapping.csv does not cover yet: {len(rows)}")
    for cat in cats:
        group = [r for r in rows if r["category"] == cat]
        if group:
            print(f"  ## {cat} ({len(group)})")
            for r in group[:PRINT_LIMIT]:
                print(f"  {r['count']:>4}  {r['candidate'][:40]:<40}  p.{r['pages']:<10} …{r['context'][:90]}…")
            if len(group) > PRINT_LIMIT:
                print(f"        ... {len(group) - PRINT_LIMIT} more in {save_to}")
    with open(save_to, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=["category", "candidate", "count", "pages", "context"])
        w.writeheader()
        w.writerows(rows)
    return rows


def extra_scan(text, mapping):
    """Safety net: emails / URLs / codes still in the output that are not one of your replacements."""
    replacements = " ".join(m["replacement"] for m in mapping)
    for cat, rx, grp in REGEX_DETECTORS:
        if cat in ("email", "url") or cat.startswith("code"):
            hits = sorted({m.group(grp) for m in rx.finditer(text)} - {""})
            hits = [h for h in hits if h not in replacements]
            if hits:
                print(f"  still present - {cat}: {hits[:10]}{' ...' if len(hits) > 10 else ''}")


def run_steps_2_to_4(doc, coefficient=None, mapping_path=MAPPING_CSV):
    if not Path(mapping_path).exists():
        print(f"{mapping_path} not found. Fill it in (see {MAPPING_TEMPLATE_CSV}) and run this again.")
        return None
    print("Step 2 - reading mapping")
    mapping = read_mapping(mapping_path)
    print(f"  {len(mapping)} mapping rows")
    coef = ask_coefficient(coefficient)

    # Step 3 runs on a fresh copy of the pages BEFORE text extraction, because it needs the ix tags.
    pages = {n: copy.deepcopy(el) for n, el in doc["pages"].items()}
    records, skipped = scale_numbers(pages, coef, doc["currency_units"], doc["per_share_units"])
    texts = prepared_texts(pages)              # hyphenation joined, signature blocks + service lines removed
    unmapped_candidates(texts, mapping)        # names / products not in mapping.csv yet (nothing changed)
    text = assemble_text(texts)

    # Step 2 runs on the extracted text, so names split across pdf2htmlEX spans are still found.
    text, counts = replace_names(text, mapping)
    print("\nStep 2 - names replaced")
    for m in mapping:
        c = counts[m["original"]]
        print(f"  {c:>4} × {m['original']}  →  {m['replacement']}" + ("   (0 hits: misspelled, or covered by a longer entry)" if not c else ""))

    print(f"\nStep 3 - scaled {len(records)} currency amounts by {coef}")
    for r in records[:8]:
        print(f"  p.{r['page']}  {r['original_text']:>15}  →  {r['scaled_text']:<15}  {r['concept']}")
    for reason, c in skipped.most_common():
        print(f"  not changed: {c} × {reason}")
    print("  note: scaled amounts are rounded one by one, so totals can differ from sums by ±1.")

    print("\nStep 4 - final check (case-insensitive, no word boundaries, 60 characters of context)")
    leaks = leak_check(text, mapping)
    leaks += extra_term_check(text, mapping)
    leaks += squashed_leak_check(text, mapping)
    extra_scan(text, mapping)

    first = text.split("\n\n=== Page ")[0].split("\n")
    print(f"\nFirst 15 lines of {first[0].strip('= ')} in the result:")
    for line in first[1:16]:
        print(f"  | {line}")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    out_txt, out_key = OUTPUT_DIR / "anonymized.txt", OUTPUT_DIR / "key.json"
    if Path(doc["source"]).resolve() in (out_txt.resolve(), out_key.resolve()):
        raise RuntimeError("Output path equals the input file - refusing to overwrite the original.")
    out_txt.write_text(text, encoding="utf-8")
    key = {"created": time.strftime("%Y-%m-%dT%H:%M:%S"), "source_file": doc["source"],
           "pages": list(doc["pages"]), "coefficient": str(coef), "mapping": mapping, "numbers": records}
    out_key.write_text(json.dumps(key, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"\nSaved {out_txt} and {out_key}  (key.json contains the real names - keep it private).")
    if leaks:
        print("  ! Fix the remaining matches (add rows to mapping.csv) before sharing anonymized.txt.")
    return text


def deanonymize(text=None, key_path=OUTPUT_DIR / "key.json", min_digits=3):
    """Reverse names and numbers.
    Names: replacement → original (longest first, whole words, possessives kept).
    Numbers: every scaled amount recorded in key.json → original, i.e. scaled ÷ coefficient with the
    original rounding restored exactly. Tokens with fewer than `min_digits` digits are left alone because
    they could be note numbers. For numbers you typed yourself use unscale(x).
    With text=None it reads output/anonymized.txt and writes output/deanonymized.txt."""
    key_path = Path(key_path)
    key = json.loads(key_path.read_text(encoding="utf-8"))
    write = text is None
    if text is None:
        text = (key_path.parent / "anonymized.txt").read_text(encoding="utf-8")

    rev = {}
    for m in sorted(key["mapping"], key=lambda m: len(m["original"]), reverse=True):
        rev.setdefault(_nows(m["replacement"]), (m["replacement"], m["original"]))
    n_names = Counter()
    if rev:
        def _name(mo):
            repl, orig = rev[_nows(mo.group(1))]
            n_names[repl] += 1
            return orig + (mo.group(2) or "")
        text = build_name_regex(r for r, _ in rev.values()).sub(_name, text)

    back, ambiguous = {}, set()
    for r in key["numbers"]:
        s, o = r["scaled_text"], r["original_text"]
        if back.get(s, o) != o:
            ambiguous.add(s)
        back[s] = o
    back = {s: o for s, o in back.items()
            if s not in ambiguous and s != o and sum(ch.isdigit() for ch in s) >= min_digits}
    n_nums = 0
    if back:
        alts = "|".join(re.escape(s) for s in sorted(back, key=len, reverse=True))
        rx = re.compile(rf"(?<![\d.,])(?:{alts})(?!\d|[.,]\d)")

        def _num(mo):
            nonlocal n_nums
            n_nums += 1
            return back[mo.group(0)]
        text = rx.sub(_num, text)

    print(f"deanonymize: {sum(n_names.values())} names and {n_nums} amounts restored"
          + (f"; {len(ambiguous)} ambiguous amounts left as is" if ambiguous else ""))
    if write:
        (key_path.parent / "deanonymized.txt").write_text(text, encoding="utf-8")
        print(f"Saved {key_path.parent / 'deanonymized.txt'}")
    return text


def unscale(x, key_path=OUTPUT_DIR / "key.json"):
    """Any scaled number → original: x ÷ coefficient (e.g. unscale(1433264) or unscale('1,433,264'))."""
    coef = Decimal(json.loads(Path(key_path).read_text(encoding="utf-8"))["coefficient"])
    return Decimal(str(x).replace(",", "").replace(" ", "")) / coef


def compare_numbers(doc, page, n=10, text=None, key_path=OUTPUT_DIR / "key.json"):
    """First n numbers of one page: original report (same cleanup, nothing replaced) vs. the result.
    Pseudonyms that contain digits ("Person 1") are ignored on both sides so the lists stay aligned."""
    if text is None:
        text = (OUTPUT_DIR / "anonymized.txt").read_text(encoding="utf-8")
    mapping = json.loads(Path(key_path).read_text(encoding="utf-8"))["mapping"] if Path(key_path).exists() else []
    parts = re.split(r"(?m)^=== Page (\d+) ===$", text)
    out_page = dict(zip(parts[1::2], parts[2::2])).get(str(page))
    if out_page is None:
        print(f"Page {page} is not in the result.")
        return
    orig_page = prepared_texts(doc["pages"], verbose=False)[page]
    for m in mapping:
        out_page = out_page.replace(m["replacement"], " ")
        orig_page = orig_page.replace(m["original"], " ")
    num = re.compile(r"\d(?:[\d,.]*\d)?")
    a, b = num.findall(orig_page)[:n], num.findall(out_page)[:n]
    tagged = any(is_ix(d, "nonfraction") for d in doc["pages"][page].iter())
    print(f"Page {page} ({'has' if tagged else 'no'} ix:nonFraction tags) - first {n} numbers")
    print(f"  {'#':>2}  {'original':>15}  {'output':>15}")
    for i in range(max(len(a), len(b))):
        x, y = (a[i] if i < len(a) else "-"), (b[i] if i < len(b) else "-")
        print(f"  {i + 1:>2}  {x:>15}  {y:>15}  {'same' if x == y else 'DIFFERENT'}")
    print(f"'[AMOUNT]' in the whole result: {text.count('[AMOUNT]')}")


In [ ]:
# 5. RUN Step 1 - loads INPUT_FILE / PAGES, writes candidates.csv and mapping_template.csv
doc = prepare_document(INPUT_FILE, PAGES)
candidates = run_step1(doc)


In [ ]:
# 6. RUN Steps 2-4 - after you have filled in and saved MAPPING_CSV
if not IN_NOTEBOOK:
    input(f"\nEdit {MAPPING_CSV} now, then press Enter to continue...")
anonymized_text = run_steps_2_to_4(doc, COEFFICIENT, mapping_path=MAPPING_CSV)


In [ ]:
# 7. (Later) reverse the process
# restored = deanonymize()                      # reads output/anonymized.txt
# restored = deanonymize("text with Company A and 1,433,264")
# unscale("1,433,264")
# compare_numbers(doc, 9)                       # first 10 numbers of page 9: original vs. result
